# Data collection

Collect high-engagement Met Gala YouTube video IDs, then fetch all available comments and replies for those videos.


In [1]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv
from googleapiclient.discovery import build

load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
VIDEO_IDS_FILE = DATA_DIR / "video_ids.json"
COMMENTS_FILE = DATA_DIR / "video_data.json"

In [2]:

MIN_VIDEO_COMMENTS = 50 # min comments per selected video id 
MAX_HIGH_ENGAGEMENT_VIDEOS = 200 # max videos pulled post-engagement filter
MAX_VIDEOS_PER_QUERY = 50 # youtube defined maximum videos per query
BATCH_SIZE = 50 # youtube defined maximum videos per batch
API_MAX_RESULTS = 100 # youtube defined maximum comments per query
COMMENTS_ORDER = "time" # time or relevance

PUBLISHED_AFTER = datetime(2026, 5, 3, tzinfo=timezone.utc)
PUBLISHED_BEFORE = datetime(2026, 5, 9, 23, 59, 59, tzinfo=timezone.utc)


In [3]:

met_gala_queries = [
    "Met Gala 2026",
    "Met Gala 2026 live",
    "Met Gala 2026 reaction",
    "Met Gala 2026 review",
    "Met Gala 2026 looks",
    "Met Gala 2026 theme",
]

In [4]:

api_key = os.getenv("YOUTUBE_API_KEY")
if not api_key:
    raise ValueError("Missing YOUTUBE_API_KEY. Add it to .env before running collection.")
youtube_client = build("youtube", "v3", developerKey=api_key)
print("YouTube client initialised")


YouTube client initialised


In [ ]:
def fetch_high_engagement_video_ids(client, queries, output_file=VIDEO_IDS_FILE):
    candidate_ids = set()

    for query in queries:
        print(f"Searching: {query}")
        response = client.search().list(
            q=query,
            part="snippet",
            type="video",
            maxResults=MAX_VIDEOS_PER_QUERY,
        ).execute()

        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            title_description = f"{snippet.get('title', '')} {snippet.get('description', '')}".lower()
            published_at = datetime.fromisoformat(snippet["publishedAt"].replace("Z", "+00:00"))

            if "met gala" not in title_description and "metgala" not in title_description:
                continue
            if not (PUBLISHED_AFTER <= published_at <= PUBLISHED_BEFORE):
                continue

            candidate_ids.add(item["id"]["videoId"])

    video_metadata = {}
    candidate_ids = list(candidate_ids)

    for i in range(0, len(candidate_ids), BATCH_SIZE):
        batch = candidate_ids[i:i + BATCH_SIZE]
        response = client.videos().list(
            id=",".join(batch),
            part="snippet,statistics",
        ).execute()

        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            stats = item.get("statistics", {})
            comment_count = int(stats.get("commentCount", 0))

            if comment_count < MIN_VIDEO_COMMENTS:
                continue

            video_metadata[item["id"]] = {
                "videoId": item["id"],
                "title": snippet.get("title", ""),
                "channelId": snippet.get("channelId", ""),
                "channelTitle": snippet.get("channelTitle", ""),
                "publishedAt": snippet.get("publishedAt", ""),
                "viewCount": int(stats.get("viewCount", 0)),
                "likeCount": int(stats.get("likeCount", 0)),
                "commentCount": comment_count,
            }

    video_ids = sorted(
        video_metadata,
        key=lambda video_id: (
            video_metadata[video_id]["commentCount"],
            video_metadata[video_id]["likeCount"],
            video_metadata[video_id]["viewCount"],
        ),
        reverse=True,
    )[:MAX_HIGH_ENGAGEMENT_VIDEOS]

    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with output_file.open("w", encoding="utf-8") as f:
        json.dump(
            {
                "queries": queries,
                "minVideoComments": MIN_VIDEO_COMMENTS,
                "maxHighEngagementVideos": MAX_HIGH_ENGAGEMENT_VIDEOS,
                "videoIds": video_ids,
                "videos": {video_id: video_metadata[video_id] for video_id in video_ids},
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    print(f"Saved top {len(video_ids)} high-engagement video IDs to {output_file}")
    return video_ids


video_ids = fetch_high_engagement_video_ids(youtube_client, met_gala_queries)


Searching: Met Gala 2026
Searching: Met Gala 2026 live


In [ ]:
def fetch_comments_and_replies(client, video_ids_file=VIDEO_IDS_FILE, output_file=COMMENTS_FILE):
    with Path(video_ids_file).open("r", encoding="utf-8") as f:
        video_lookup = json.load(f)

    video_ids = video_lookup["videoIds"]
    metadata_by_video = video_lookup["videos"]
    videos = []
    total_records = 0
    total_replies = 0

    for index, video_id in enumerate(video_ids, start=1):
        comments = []
        next_comment_page = None
        print(f"[{index}/{len(video_ids)}] Fetching comments for {video_id}")

        while True:
            request = {
                "videoId": video_id,
                "part": "snippet",
                "maxResults": API_MAX_RESULTS,
                "textFormat": "plainText",
                "order": COMMENTS_ORDER,
            }
            if next_comment_page:
                request["pageToken"] = next_comment_page

            response = client.commentThreads().list(**request).execute()

            for thread in response.get("items", []):
                thread_snippet = thread.get("snippet", {})
                top_comment = thread_snippet.get("topLevelComment", {})
                top_snippet = top_comment.get("snippet", {})
                parent_comment_id = top_comment.get("id")
                parent_author_id = top_snippet.get("authorChannelId", {}).get("value")

                comments.append({
                    "commentId": parent_comment_id,
                    "authorId": parent_author_id,
                    "author": top_snippet.get("authorDisplayName"),
                    "text": top_snippet.get("textDisplay", ""),
                    "publishedAt": top_snippet.get("publishedAt"),
                    "updatedAt": top_snippet.get("updatedAt"),
                    "likeCount": top_snippet.get("likeCount", 0),
                    "totalReplyCount": thread_snippet.get("totalReplyCount", 0),
                    "parentCommentId": None,
                    "replyToAuthorId": None,
                    "videoId": video_id,
                    "isReply": False,
                })

                if thread_snippet.get("totalReplyCount", 0) > 0:
                    next_reply_page = None
                    while True:
                        reply_request = {
                            "parentId": parent_comment_id,
                            "part": "snippet",
                            "maxResults": API_MAX_RESULTS,
                            "textFormat": "plainText",
                        }
                        if next_reply_page:
                            reply_request["pageToken"] = next_reply_page

                        reply_response = client.comments().list(**reply_request).execute()

                        for reply in reply_response.get("items", []):
                            reply_snippet = reply.get("snippet", {})
                            comments.append({
                                "commentId": reply.get("id"),
                                "authorId": reply_snippet.get("authorChannelId", {}).get("value"),
                                "author": reply_snippet.get("authorDisplayName"),
                                "text": reply_snippet.get("textDisplay", ""),
                                "publishedAt": reply_snippet.get("publishedAt"),
                                "updatedAt": reply_snippet.get("updatedAt"),
                                "likeCount": reply_snippet.get("likeCount", 0),
                                "totalReplyCount": 0,
                                "parentCommentId": parent_comment_id,
                                "replyToAuthorId": parent_author_id,
                                "videoId": video_id,
                                "isReply": True,
                            })

                        next_reply_page = reply_response.get("nextPageToken")
                        if not next_reply_page:
                            break

            next_comment_page = response.get("nextPageToken")
            if not next_comment_page:
                break

        reply_count = sum(1 for comment in comments if comment["isReply"])
        total_records += len(comments)
        total_replies += reply_count
        print(f"  saved {len(comments)} comments/replies, including {reply_count} replies")

        metadata = metadata_by_video[video_id]
        videos.append({
            "title": metadata.get("title", ""),
            "videoId": video_id,
            "channelId": metadata.get("channelId", ""),
            "channelTitle": metadata.get("channelTitle", ""),
            "publishedAt": metadata.get("publishedAt", ""),
            "viewCount": metadata.get("viewCount", 0),
            "likeCount": metadata.get("likeCount", 0),
            "commentCount": metadata.get("commentCount", 0),
            "comments": comments,
        })

    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    with output_file.open("w", encoding="utf-8") as f:
        json.dump({"videos": videos}, f, ensure_ascii=False, indent=2)

    print(f"Saved {total_records} comments/replies, including {total_replies} replies, to {output_file}")
    return videos

videos = fetch_comments_and_replies(youtube_client)
